In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
import pinecone
from sentence_transformers import SentenceTransformer

C:\Users\akhil\Langgraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
files = pd.read_csv("course_descriptions.csv", encoding = "ANSI")

In [3]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]}, the slug is {row["course_slug"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''

In [4]:
pd.set_option('display.max_rows', 106)
files['course_description_new'] = files.apply(create_course_description, axis = 1)
print(files["course_description_new"])

0      The course name is Introduction to Tableau, th...
1      The course name is The Complete Data Visualiza...
2      The course name is Introduction to R Programmi...
3      The course name is Data Preprocessing with Num...
4      The course name is Introduction to Data and Da...
5      The course name is Data Cleaning and Preproces...
6      The course name is Introduction to Business An...
7      The course name is Data Analysis with Excel Pi...
8      The course name is SQL, the slug is sql,\n    ...
9      The course name is Credit Risk Modeling in Pyt...
10     The course name is Python Programmer Bootcamp,...
11     The course name is SQL + Tableau + Python, the...
12     The course name is Introduction to Jupyter, th...
13     The course name is Statistics, the slug is sta...
14     The course name is Mathematics, the slug is ma...
15     The course name is Introduction to Excel, the ...
16     The course name is Probability, the slug is pr...
17     The course name is Start

In [5]:
load_dotenv(find_dotenv(), override = True)
pc = Pinecone(
   api_key= os.getenv("PINECONE_API_KEY")
)

In [6]:
index_name = "my-index"
metric = "cosine"

In [7]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

my-index succesfully deleted.


In [8]:
#model = SentenceTransformer("all-MiniLM-L6-v2")
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

C:\Users\akhil\Langgraph\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akhil\.cache\huggingface\hub\models--sentence-transformers--multi-qa-distilbert-cos-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\akhil\Langgraph\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: Us

In [9]:
pc.create_index(
    name = index_name,
    dimension = model.get_embedding_dimension(),
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1")
    )

Name:,my-index
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://my-index-rff7cv2.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [10]:
index = pc.Index(index_name)

## Creating Embeddings

In [11]:
def create_embeddings(row):
    combined_text = ' '.join([str(row[field]) for field in ['course_description', 'course_description_new', 'course_description_short']])
    embedding = model.encode(combined_text, show_progress_bar = False)
    return embedding

In [12]:
files["embedding"] = files.apply(create_embeddings, axis = 1)

In [13]:
vectors_to_upsert = [(str(row["course_name"]), row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("Data upserted to Pinecone index")

Data upserted to Pinecone index


## Semantic Search

In [18]:
query = "clustering"
query_embedding = model.encode( query, show_progress_bar = False).tolist()

In [19]:
query_result =  index.query(
    vector= [query_embedding],
    top_k = 5,
    include_values= True
)

In [22]:
for match in query_result['matches']:
    print(f''' Match id {match['id']},  match score {match['score']}
    ''')

 Match id Machine Learning in Excel,  match score 0.301053017
    
 Match id Machine Learning with K-Nearest Neighbors,  match score 0.241065025
    
 Match id Machine Learning in Python,  match score 0.19934082
    
 Match id Growth Analysis with SQL, Python, and Tableau  ,  match score 0.172528267
    
 Match id Introduction to R Programming,  match score 0.157257065
    
